# Anomaly Detection With Causal Models

In [1]:
import numpy as np
import pandas as pd

import dowhy
from dowhy import gcm

# (USE ALL CORES TO SPEED UP COMPUTATIONS)
gcm.config.set_default_n_jobs(n_jobs=-1)

gcm.config.enable_progress_bars()

import networkx as nx
from networkx.drawing.nx_pydot import read_dot

from sklearn.linear_model import LinearRegression

import scipy

# Set Seed for Reproducability

In [2]:
SEED = 432432
np.random.seed(SEED)

# Dataset
from: https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15

In [3]:
df_training = pd.read_csv('data/UNSW-NB15/UNSW_NB15_training-set.csv')
df_testing = pd.read_csv('data/UNSW-NB15/UNSW_NB15_testing-set.csv')

In [4]:
selected_cols = list(df_training.select_dtypes(include=['int', 'float']).columns)

#df_train_selected = df_training[selected_cols+['label', 'attack_cat']].copy()
df_train_selected = df_training[selected_cols+['attack_cat']].copy()
df_train_selected['attack_cat'] = df_train_selected['attack_cat'].astype('category').cat.codes
#df_train_selected = df_training[selected_cols]

df_train_selected.head()

,id,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label,attack_cat
0,1,0.000011,2,0,496,0,90909.0902,254,0,180363632.0,...,1,2,0,0,0,1,2,0,0,6
1,2,0.000008,2,0,1762,0,125000.0003,254,0,881000000.0,...,1,2,0,0,0,1,2,0,0,6
2,3,0.000005,2,0,1068,0,200000.0051,254,0,854400000.0,...,1,3,0,0,0,1,3,0,0,6
3,4,0.000006,2,0,900,0,166666.6608,254,0,600000000.0,...,1,3,0,0,0,2,3,0,0,6
4,5,0.000010,2,0,2126,0,100000.0025,254,0,850400000.0,...,1,3,0,0,0,2,3,0,0,6


# Model

In [5]:
causal_graph_networkx_format = read_dot('CD_AllNumerical.gv')
print(causal_graph_networkx_format.nodes())
#print(causal_graph_networkx_format.edges())

['id', 'dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports', 'label', 'attack_cat']


In [6]:
causal_model = gcm.InvertibleStructuralCausalModel(graph=causal_graph_networkx_format)

## Setting Causal Mechanisms

In [7]:
# (AUTO)
# (NOTE: THIS IS COMPUTE INTENSIVE!)
#auto_assignment_summary = gcm.auto.assign_causal_mechanisms(causal_model, df_train_selected)

# (PRINT THE ASSIGNED PIPELINES)
#print(auto_assignment_summary)

In [8]:
# (MANUAL)
for node in causal_graph_networkx_format.nodes():
    if dowhy.graph.is_root_node(causal_model.graph, node):
        causal_model.set_causal_mechanism(node, gcm.ScipyDistribution(scipy.stats.halfnorm))
        print(f'ROOT: {node}')
    else:
        print(node)
        causal_model.set_causal_mechanism(node, gcm.AdditiveNoiseModel(gcm.ml.SklearnRegressionModel(LinearRegression())))

id
dur
spkts
dpkts
ROOT: sbytes
dbytes
rate
sttl
dttl
sload
dload
sloss
dloss
sinpkt
dinpkt
sjit
djit
swin
stcpb
dtcpb
dwin
tcprtt
synack
ackdat
smean
dmean
trans_depth
ROOT: response_body_len
ct_srv_src
ct_state_ttl
ct_dst_ltm
ct_src_dport_ltm
ct_dst_sport_ltm
ct_dst_src_ltm
ROOT: is_ftp_login
ct_ftp_cmd
ct_flw_http_mthd
ct_src_ltm
ct_srv_dst
ROOT: is_sm_ips_ports
label
attack_cat


## Fit to Data

In [9]:
gcm.fit(causal_model, df_train_selected)

Fitting causal mechanism of node attack_cat: 100%|█████████████████████████████████████| 42/42 [00:00<00:00, 47.79it/s]


# Anomaly Attribution

In [10]:
SAMPLE_IDX = 0

# +1 REQUIRED FOR SINGLE SAMPLES TO GET VALID DATAFRAME!
df_anomalous_data = df_train_selected.copy().iloc[SAMPLE_IDX:(1+SAMPLE_IDX), :]

#df_anomalous_data['dload'] = 2*df_anomalous_data['dload']
df_anomalous_data['attack_cat'] = 6

In [11]:
df_anomalous_data.head()

,id,dur,spkts,dpkts,sbytes,dbytes,rate,sttl,dttl,sload,...,ct_dst_sport_ltm,ct_dst_src_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,ct_src_ltm,ct_srv_dst,is_sm_ips_ports,label,attack_cat
0,1,0.000011,2,0,496,0,90909.0902,254,0,180363632.0,...,1,2,0,0,0,1,2,0,0,6


In [12]:
attribution_scores = gcm.attribute_anomalies(causal_model, 'label', anomaly_samples=df_anomalous_data)
#attribution_scores = gcm.attribute_anomalies(causal_model, 'attack_cat', anomaly_samples=df_anomalous_data)

Evaluating set functions...: 100%|███████████████████████████████████████████████████| 403/403 [00:27<00:00, 14.82it/s]


## List Attribution Scores

In [13]:
for k, v in attribution_scores.items():
    print(f'{k:<20} {v.item():10.4f}')

is_ftp_login            -0.1197
is_sm_ips_ports         -0.1058
synack                  -0.0301
ct_dst_sport_ltm         0.0799
ct_ftp_cmd              -0.0001
ct_flw_http_mthd        -0.0154
tcprtt                   0.0725
ct_src_dport_ltm         0.0428
trans_depth             -0.0056
ackdat                   0.0000
dttl                    -0.0567
dwin                    -0.1051
swin                     0.1582
ct_state_ttl            -0.0407
ct_dst_ltm               0.0555
ct_src_ltm               0.0677
ct_dst_src_ltm           0.1976
label                    0.8894


## Sorted List

In [14]:
attribution_scores_sorted = {k: v for k, v in sorted(attribution_scores.items(), key=lambda x: abs(x[1]), reverse=True)}

for k, v in attribution_scores_sorted.items():
    print(f'{k:<20} {v.item():10.4f}')

label                    0.8894
ct_dst_src_ltm           0.1976
swin                     0.1582
is_ftp_login            -0.1197
is_sm_ips_ports         -0.1058
dwin                    -0.1051
ct_dst_sport_ltm         0.0799
tcprtt                   0.0725
ct_src_ltm               0.0677
dttl                    -0.0567
ct_dst_ltm               0.0555
ct_src_dport_ltm         0.0428
ct_state_ttl            -0.0407
synack                  -0.0301
ct_flw_http_mthd        -0.0154
trans_depth             -0.0056
ct_ftp_cmd              -0.0001
ackdat                   0.0000


# Sampling

# Interventional Examples

In [15]:
interventional_samples = gcm.interventional_samples(
    causal_model,
    {'attack_cat': lambda x: 1},
    num_samples_to_draw=1000
)
interventional_samples.head()

,sbytes,response_body_len,is_ftp_login,is_sm_ips_ports,dbytes,dload,synack,ct_dst_sport_ltm,ct_ftp_cmd,ct_flw_http_mthd,...,attack_cat,id,spkts,smean,rate,dur,dmean,sload,sttl,djit
0,67767.911171,65801.024351,0.030910,0.155344,100562.481623,89900.615730,0.000060,3.918782,0.031178,-0.003997,...,1,46457.126562,-35.677895,55.945995,152657.257265,2.094069,17.798346,9.941489e+07,130.432117,290.879532
1,127983.528506,5799.735853,0.084385,0.020812,8832.426689,-32468.268193,0.049282,17.778273,0.085117,-0.010911,...,1,-27638.158411,58.215899,52.569077,19413.372790,2.018626,132.105505,2.182253e+07,119.825020,143.045215
2,148456.087917,14174.889428,0.162118,0.003966,21586.961005,-57528.843253,0.000815,0.574025,0.163524,-0.020961,...,1,12814.961715,-3.045937,206.505118,128693.183784,0.941989,-21.088193,7.559088e+07,167.609552,-14.066352
3,160804.782913,8703.772833,0.128006,0.042299,13340.989092,-44770.782224,0.000250,3.663658,0.129115,-0.016551,...,1,44827.947325,56.203295,53.922094,82674.009790,-2.121888,377.850418,8.220499e+07,99.563305,290.961052
4,84927.952917,14455.725663,0.042040,0.067009,22282.646940,16439.957439,0.000082,1.889538,0.042404,-0.005436,...,1,58396.480551,6.196880,144.357479,124310.167395,-0.053179,150.611347,6.751739e+07,213.442895,191.572999


In [16]:
interventional_samples2 = gcm.interventional_samples(
    causal_model,
    {'attack_cat': lambda x: 3},
    num_samples_to_draw=1000
)
interventional_samples2.head()

,sbytes,response_body_len,is_ftp_login,is_sm_ips_ports,dbytes,dload,synack,ct_dst_sport_ltm,ct_ftp_cmd,ct_flw_http_mthd,...,attack_cat,id,spkts,smean,rate,dur,dmean,sload,sttl,djit
0,291521.007249,59347.894867,0.093756,0.030049,90381.000764,51100.500329,0.055084,0.753650,0.094569,-0.012122,...,3,101571.095779,-206.694427,81.961686,-233123.336341,-0.626700,191.431704,-1.305736e+08,176.464640,186.736669
1,61809.472530,67562.753465,0.061108,0.192505,111715.421612,77780.078549,0.000119,3.839436,0.061638,-0.007901,...,3,30487.338237,-10.153676,-80.225956,-87116.861410,1.668071,45.459066,3.101987e+08,176.649362,288.404083
2,301834.031036,12980.770379,0.026579,0.130055,19768.435261,20455.481666,0.000052,17.930162,0.026809,-0.003437,...,3,51763.958300,-25.670370,214.751613,353841.823577,0.031754,-32.536360,2.349038e+08,176.013704,205.841469
3,318489.001806,46014.322012,0.058480,0.015924,70075.282068,45764.671761,0.082511,1.846340,0.058987,-0.007561,...,3,69319.196155,11.837328,89.362718,-77639.425659,1.372258,63.016939,-6.080478e+07,323.369341,248.399877
4,204000.492499,42921.882562,0.104240,0.149608,80661.801252,17054.622487,0.132741,0.726104,0.105144,-0.013478,...,3,84461.579544,-19.318560,84.237934,969648.442146,1.193161,95.914312,7.123018e+08,170.575643,841.911574


In [17]:
def check_difference(df1, df2):
    print(f'df1.shape: {df1.shape}, df2.shape: {df2.shape}')
    diff_locations = (df1 != df2)
    differences = pd.DataFrame({
        'row': diff_locations.stack().loc[lambda x: x].index.get_level_values(0),
        'column': diff_locations.stack().loc[lambda x: x].index.get_level_values(1),
        'df1': df1.stack().loc[lambda x: diff_locations.stack()],
        'df2': df2.stack().loc[lambda x: diff_locations.stack()]
    })
    
    print(differences)

In [18]:
check_difference(interventional_samples, interventional_samples2)

df1.shape: (1000, 42), df2.shape: (1000, 42)
                       row             column           df1           df2
0   sbytes               0             sbytes  6.776791e+04  2.915210e+05
    response_body_len    0  response_body_len  6.580102e+04  5.934789e+04
    is_ftp_login         0       is_ftp_login  3.090989e-02  9.375601e-02
    is_sm_ips_ports      0    is_sm_ips_ports  1.553443e-01  3.004894e-02
    dbytes               0             dbytes  1.005625e+05  9.038100e+04
...                    ...                ...           ...           ...
999 dur                999                dur -1.490260e-01  2.652893e+00
    dmean              999              dmean  1.639808e+02  3.420476e+02
    sload              999              sload  1.418505e+08  1.595355e+08
    sttl               999               sttl  1.077423e+02  1.591682e+02
    djit               999               djit  1.301028e+03  9.799085e+02

[42000 rows x 4 columns]


# Conterfactual Examples

In [19]:
def compare_columns(df1, df2, tolerance=1e-5):
    if df1.shape != df2.shape:
        raise ValueError('DataFrames must have the same shape')

    diff = (df2 - df1).abs()
    mask = diff > tolerance
    altered_columns = mask.any()
    altered_columns = altered_columns[altered_columns].index.tolist()

    if not altered_columns:
        return [], pd.Series(dtype=float)

    pct_change = diff / df1.replace(0, np.nan).abs() * 100
    pct_change[~mask] = 0
    avg_pct_change = pct_change[altered_columns].mean()
    avg_pct_change = avg_pct_change.sort_values(ascending=False)
    return altered_columns, avg_pct_change
    

def check_cf_difference(_causal_model, _intervention0, _intervention1, _observed_data):
    def normalize(series):
        # (BASIC MIN-MAX SCALER TO FIT ANY VALUE INTO [0.0, 1.0])
        return series / series.max()

    # CROSS COUNTERFACTUAL SAMPLES (DIFFERENCE BETWEEN TWO SETS OF SAMPLES)
    # (USE USE COUNTERFACTUAL SAMPLES TO AVIOD MODEL BASED ERROR)
    # (ERRORS SHOULD CANCEL EACH OTHER)
    counterfactual_samples_label0 = gcm.counterfactual_samples(
        _causal_model,
        interventions=_intervention0,
        observed_data=_observed_data
    )
    
    counterfactual_samples_label1 = gcm.counterfactual_samples(
        _causal_model,
        interventions=_intervention1,   
        observed_data=_observed_data
    )

    altered_columns, avg_pct_chang = compare_columns(counterfactual_samples_label0, counterfactual_samples_label1)
    #print('Altered Columns:', altered_columns)
    print('Average % Change per Column:')
    print(normalize(avg_pct_chang).map('{:.4f}'.format))

## Change Label

In [20]:
# (CONSIDER ONLY ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==1].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'label': lambda x: 0},
    _intervention1={'label': lambda x: 1},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
5    18871
3    11132
4     6062
2     4089
7     3496
0      677
1      583
8      378
9       44
Name: count, dtype: int64
Average % Change per Column:
dmean         1.0000
djit          0.6179
dur           0.3951
rate          0.3911
sload         0.2044
attack_cat    0.1698
sttl          0.0861
smean         0.0472
spkts         0.0153
label            nan
dtype: object


In [21]:
# (CONSIDER ONLY NON-ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==0].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'label': lambda x: 0},
    _intervention1={'label': lambda x: 1},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
6    37000
Name: count, dtype: int64
Average % Change per Column:
djit          1.0000
dmean         0.0309
dur           0.0001
sload         0.0000
rate          0.0000
sttl          0.0000
attack_cat    0.0000
smean         0.0000
spkts         0.0000
label            nan
dtype: object


## Change Attack Type
### Anomaly Case

In [22]:
# (CONSIDER ONLY ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==1].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 0},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
5    18871
3    11132
4     6062
2     4089
7     3496
0      677
1      583
8      378
9       44
Name: count, dtype: int64
Average % Change per Column:
dur           1.0000
djit          0.5835
smean         0.4704
dmean         0.4247
sload         0.1278
rate          0.0839
attack_cat    0.0799
sttl          0.0257
spkts         0.0074
dtype: object


In [23]:
# (CONSIDER ONLY ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==1].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 1},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
5    18871
3    11132
4     6062
2     4089
7     3496
0      677
1      583
8      378
9       44
Name: count, dtype: int64
Average % Change per Column:
dur           1.0000
djit          0.5835
smean         0.4704
dmean         0.4247
sload         0.1278
rate          0.0839
attack_cat    0.0799
sttl          0.0257
spkts         0.0074
dtype: object


In [24]:
# (CONSIDER ONLY ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==1].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 2},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
5    18871
3    11132
4     6062
2     4089
7     3496
0      677
1      583
8      378
9       44
Name: count, dtype: int64
Average % Change per Column:
dur           1.0000
djit          0.5835
smean         0.4704
dmean         0.4247
sload         0.1278
rate          0.0839
attack_cat    0.0799
sttl          0.0257
spkts         0.0074
dtype: object


In [25]:
# (CONSIDER ONLY ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==1].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 3},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
5    18871
3    11132
4     6062
2     4089
7     3496
0      677
1      583
8      378
9       44
Name: count, dtype: int64
Average % Change per Column:
dur           1.0000
djit          0.5835
smean         0.4704
dmean         0.4247
sload         0.1278
rate          0.0839
attack_cat    0.0799
sttl          0.0257
spkts         0.0074
dtype: object


In [26]:
# (CONSIDER ONLY ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==1].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 4},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
5    18871
3    11132
4     6062
2     4089
7     3496
0      677
1      583
8      378
9       44
Name: count, dtype: int64
Average % Change per Column:
dur           1.0000
djit          0.5835
smean         0.4704
dmean         0.4247
sload         0.1278
rate          0.0839
attack_cat    0.0799
sttl          0.0257
spkts         0.0074
dtype: object


In [27]:
# (CONSIDER ONLY ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==1].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 5},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
5    18871
3    11132
4     6062
2     4089
7     3496
0      677
1      583
8      378
9       44
Name: count, dtype: int64
Average % Change per Column:
dur           1.0000
djit          0.5835
smean         0.4704
dmean         0.4247
sload         0.1278
rate          0.0839
attack_cat    0.0799
sttl          0.0257
spkts         0.0074
dtype: object


### Non-Anomaly Case

In [28]:
# (CONSIDER ONLY NON-ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==0].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 0},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
6    37000
Name: count, dtype: int64
Average % Change per Column:
djit          1.0000
dmean         0.0384
dur           0.0008
sload         0.0000
rate          0.0000
smean         0.0000
sttl          0.0000
attack_cat    0.0000
spkts         0.0000
dtype: object


In [29]:
# (CONSIDER ONLY NON-ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==0].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 1},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
6    37000
Name: count, dtype: int64
Average % Change per Column:
djit          1.0000
dmean         0.0384
dur           0.0008
sload         0.0000
rate          0.0000
smean         0.0000
sttl          0.0000
attack_cat    0.0000
spkts         0.0000
dtype: object


In [30]:
# (CONSIDER ONLY NON-ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==0].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 2},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
6    37000
Name: count, dtype: int64
Average % Change per Column:
djit          1.0000
dmean         0.0384
dur           0.0008
sload         0.0000
rate          0.0000
smean         0.0000
sttl          0.0000
attack_cat    0.0000
spkts         0.0000
dtype: object


In [31]:
# (CONSIDER ONLY NON-ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==0].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 3},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
6    37000
Name: count, dtype: int64
Average % Change per Column:
djit          1.0000
dmean         0.0384
dur           0.0008
sload         0.0000
rate          0.0000
smean         0.0000
sttl          0.0000
attack_cat    0.0000
spkts         0.0000
dtype: object


In [32]:
# (CONSIDER ONLY NON-ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==0].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 4},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
6    37000
Name: count, dtype: int64
Average % Change per Column:
djit          1.0000
dmean         0.0384
dur           0.0008
sload         0.0000
rate          0.0000
smean         0.0000
sttl          0.0000
attack_cat    0.0000
spkts         0.0000
dtype: object


In [33]:
# (CONSIDER ONLY NON-ANOMALY CASE)
df_train_selected_counterfactual = df_train_selected[df_train_selected['label']==0].copy()
print(df_train_selected_counterfactual['attack_cat'].value_counts())

check_cf_difference(
    causal_model, 
    _intervention0={'attack_cat': lambda x: 6},
    _intervention1={'attack_cat': lambda x: 5},
    _observed_data=df_train_selected_counterfactual,
)

attack_cat
6    37000
Name: count, dtype: int64
Average % Change per Column:
djit          1.0000
dmean         0.0384
dur           0.0008
sload         0.0000
rate          0.0000
smean         0.0000
sttl          0.0000
attack_cat    0.0000
spkts         0.0000
dtype: object


# Predecessor Analysis

In [34]:
for predecessor in causal_graph_networkx_format.predecessors('label'):
    print(predecessor)

tcprtt
synack
ct_state_ttl
ct_dst_src_ltm
ct_ftp_cmd
ct_flw_http_mthd
is_sm_ips_ports


In [35]:
for predecessor in causal_graph_networkx_format.predecessors('attack_cat'):
    print(predecessor)

dloss
tcprtt
trans_depth
ct_state_ttl
ct_src_dport_ltm
ct_dst_sport_ltm
ct_dst_src_ltm
is_ftp_login
ct_flw_http_mthd
ct_src_ltm
is_sm_ips_ports
label


In [36]:
def find_the_difference_between_two_lists(l1, l2):
    return list(set(l1) - set(l2))

In [37]:
find_the_difference_between_two_lists(causal_graph_networkx_format.predecessors('attack_cat'), causal_graph_networkx_format.predecessors('label'))

['ct_dst_sport_ltm',
 'label',
 'is_ftp_login',
 'dloss',
 'ct_src_dport_ltm',
 'ct_src_ltm',
 'trans_depth']

In [38]:
find_the_difference_between_two_lists(causal_graph_networkx_format.predecessors('label'), causal_graph_networkx_format.predecessors('attack_cat'))

['ct_ftp_cmd', 'synack']

# Attributing Distributional Changes
Comapare and find the distribution difference betweeen two seperate datasets

**NOTE: THIS METHOD IS COMPUTE INTENSIVE, CONSIDER THIS LATER!**